# Fashion Image Search: Visual Product Discovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SerendipityOneInc/APIClaw-Notebook/blob/main/e-commerce/fashion_image_search_demo.ipynb)

This notebook demonstrates how to use APIClaw's **Fashion Image Search** API to find visually similar fashion products from a catalog of 200M+ items. Upload any fashion image and discover matching products across major retailers.

### What you'll learn

1. **Basic image search** — find products visually similar to a query image
2. **Bounding box cropping** — focus on a specific item in a multi-item image
3. **Text-guided image search** — combine image + text for better matching
4. **Filtered image search** — apply brand, price, and retailer filters
5. **"Shop the Look" pipeline** — detect multiple items in a street-style photo

### API Reference

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/openapi/v2/model/fashion-image-search` | POST | Search fashion products by image similarity |
| `/openapi/v2/model/fashion-product-search` | POST | Search fashion products by text (used for demo images) |

## Setup

Get a free API key (1,000 credits) at [apiclaw.io/register](https://apiclaw.io/register).

In [ ]:
!pip install -q httpx Pillow matplotlib

In [ ]:
import getpass
import os

if "APICLAW_API_KEY" not in os.environ:
    os.environ["APICLAW_API_KEY"] = getpass.getpass("Enter your APIClaw API key (hms_live_...): ")

API_KEY = os.environ["APICLAW_API_KEY"]
BASE_URL = "https://api.apiclaw.io/openapi/v2"

In [ ]:
import httpx
from PIL import Image, ImageDraw
from io import BytesIO
import matplotlib.pyplot as plt

client = httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30.0,
)


def image_search(image_url: str, limit: int = 10, **kwargs) -> dict:
    """Search fashion products by image similarity."""
    payload = {"imageUrl": image_url, "limit": limit, **kwargs}
    resp = client.post("/model/fashion-image-search", json=payload)
    resp.raise_for_status()
    return resp.json()["data"]


def text_search(query: str, top_k: int = 5) -> list[dict]:
    """Search fashion products by text (helper for getting demo images)."""
    resp = client.post("/model/fashion-product-search", json={"query": query, "topK": top_k})
    resp.raise_for_status()
    return resp.json()["data"]["products"]


def load_image(url: str) -> Image.Image:
    """Download an image from URL."""
    r = httpx.get(url, timeout=10, follow_redirects=True)
    return Image.open(BytesIO(r.content))


def show_query_and_results(query_img_url: str, results: list[dict],
                          query_title: str = "Query", max_results: int = 4,
                          bbox: list[int] | None = None):
    """Display query image alongside top matching products."""
    n = min(len(results), max_results)
    fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 5))

    # Query image
    try:
        qimg = load_image(query_img_url)
        if bbox:
            draw = ImageDraw.Draw(qimg)
            draw.rectangle(bbox, outline="red", width=3)
        axes[0].imshow(qimg)
    except Exception:
        axes[0].text(0.5, 0.5, "N/A", ha="center", va="center")
    axes[0].set_title(f"QUERY\n{query_title}", fontsize=10, fontweight="bold")
    axes[0].axis("off")

    # Result images
    for i, p in enumerate(results[:n]):
        ax = axes[i + 1]
        try:
            ax.imshow(load_image(p.get("imageUrl", "")))
        except Exception:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center")
        cat = p.get("category") or p.get("productTag") or ""
        price = p.get("price")
        score = p.get("matchScore")
        label = f"#{i+1}"
        if cat:
            label += f" | {cat}"
        if price:
            label += f" | ${price:.2f}"
        if score:
            label += f"\nscore: {score:.3f}"
        ax.set_title(label, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

---
## Task 1: Get a Query Image

First, let's use APIClaw's text search to find a fashion product. We'll use its image as the query for visual similarity search.

In [ ]:
# Find a product to use as our query image
seed_products = text_search("women red leather handbag", top_k=3)

query_product = seed_products[0]
query_image_url = query_product.get("imageUrl", "")

print(f"Query product: {query_product.get('title', '')}")
print(f"Brand: {query_product.get('brand', '-')} | Price: ${query_product.get('price', 0):.2f}")
print(f"Image URL: {query_image_url[:80]}...")

# Display
try:
    plt.figure(figsize=(4, 4))
    plt.imshow(load_image(query_image_url))
    plt.title(f"{query_product.get('title', '')[:50]}")
    plt.axis("off")
    plt.show()
except Exception as e:
    print(f"Could not display image: {e}")

---
## Task 2: Basic Image Search

Pass the product image URL to the image search endpoint and find visually similar products across the entire catalog.

In [ ]:
# Search for visually similar products
result = image_search(query_image_url, limit=8)

print(f"Query image: {result.get('imageUrl', '')[:60]}...")
print(f"Relevance score: {result.get('relevanceScore')}")
print(f"Results: {result.get('total', 0)} products")

if result.get("bbox"):
    print(f"Detected bbox: {result['bbox']}")

products = result.get("products", [])
print(f"\nTop matches:")
for i, p in enumerate(products[:5]):
    print(f"  {i+1}. {p.get('category', '-'):>12} | "
          f"score: {p.get('matchScore', 0):.3f} | "
          f"${p.get('price', 0):.2f}")

In [ ]:
# Visualize query vs top matches
show_query_and_results(query_image_url, products, query_title="Red Handbag")

---
## Task 3: Bounding Box Cropping

When an image contains multiple items (e.g., a full outfit photo), use the `bbox` parameter to focus on a specific region. The bounding box is specified as `[x1, y1, x2, y2]` in pixels.

This is the key building block for "Shop the Look" features.

In [ ]:
# Get a full-body fashion image (outfit photo)
outfit_products = text_search("women complete outfit dress shoes", top_k=3)
outfit_url = outfit_products[0].get("imageUrl", query_image_url)

# Load and measure the image to set a reasonable bbox
try:
    outfit_img = load_image(outfit_url)
    w, h = outfit_img.size
    print(f"Image size: {w}x{h}")

    # Focus on the upper half (e.g., top/jacket area)
    upper_bbox = [0, 0, w, h // 2]
    # Focus on the lower half (e.g., shoes/pants area)
    lower_bbox = [0, h // 2, w, h]

    print(f"Upper region bbox: {upper_bbox}")
    print(f"Lower region bbox: {lower_bbox}")
except Exception as e:
    print(f"Using fallback bbox: {e}")
    upper_bbox = [0, 0, 300, 300]
    lower_bbox = [0, 300, 300, 600]

In [ ]:
# Search using the upper region
upper_result = image_search(outfit_url, limit=4, bbox=upper_bbox)
upper_products = upper_result.get("products", [])

print(f"Upper region: {len(upper_products)} matches")
show_query_and_results(outfit_url, upper_products,
                       query_title="Upper Region", bbox=upper_bbox)

In [ ]:
# Search using the lower region
lower_result = image_search(outfit_url, limit=4, bbox=lower_bbox)
lower_products = lower_result.get("products", [])

print(f"Lower region: {len(lower_products)} matches")
show_query_and_results(outfit_url, lower_products,
                       query_title="Lower Region", bbox=lower_bbox)

---
## Task 4: Text-Guided Image Search

Add an `imageDescription` to guide the visual search. This helps when:
- The image contains multiple items and you want a specific one
- You want to bias results toward a particular style or color
- The image is ambiguous (e.g., a bag could be a tote or clutch)

In [ ]:
# Same image, different text guidance
descriptions = [
    None,  # baseline: image only
    "red leather crossbody bag",
    "designer evening clutch",
]

for desc in descriptions:
    kwargs = {}
    if desc:
        kwargs["imageDescription"] = desc

    result = image_search(query_image_url, limit=4, **kwargs)
    products = result.get("products", [])

    label = desc or "(no description)"
    print(f"\nDescription: \"{label}\"")
    for i, p in enumerate(products[:3]):
        print(f"  {i+1}. {p.get('category', '-'):>10} | "
              f"score: {p.get('matchScore', 0):.3f} | "
              f"${p.get('price', 0):.2f}")

In [ ]:
# Visualize how text guidance changes results
fig, axes = plt.subplots(len(descriptions), 5, figsize=(18, 4 * len(descriptions)))

for row, desc in enumerate(descriptions):
    kwargs = {}
    if desc:
        kwargs["imageDescription"] = desc
    result = image_search(query_image_url, limit=4, **kwargs)
    products = result.get("products", [])

    # Query
    try:
        axes[row, 0].imshow(load_image(query_image_url))
    except Exception:
        pass
    axes[row, 0].set_title(f"Query\n\"{desc or 'image only'}\"", fontsize=9, fontweight="bold")
    axes[row, 0].axis("off")

    # Results
    for j, p in enumerate(products[:4]):
        ax = axes[row, j + 1]
        try:
            ax.imshow(load_image(p.get("imageUrl", "")))
        except Exception:
            pass
        score = p.get("matchScore", 0)
        cat = p.get("category", "")
        ax.set_title(f"{cat}\nscore: {score:.3f}", fontsize=8)
        ax.axis("off")

plt.suptitle("Effect of Text Guidance on Image Search", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Task 5: Filtered Image Search

Combine visual search with structured filters: brands, price range, and retailer domains.

In [ ]:
# Image search with price filter: find affordable alternatives
affordable = image_search(
    query_image_url,
    limit=5,
    priceMin=20,
    priceMax=100,
)

print(f"Affordable alternatives ($20-$100): {affordable.get('total', 0)} results")
affordable_products = affordable.get("products", [])
for i, p in enumerate(affordable_products):
    print(f"  {i+1}. ${p.get('price', 0):>6.2f} | {p.get('category', '-')} | score: {p.get('matchScore', 0):.3f}")

show_query_and_results(query_image_url, affordable_products, query_title="Find Affordable Dupes")

In [ ]:
# Image search with brand filter
branded = image_search(
    query_image_url,
    limit=5,
    brands=["Coach", "Michael Kors", "Kate Spade"],
)

branded_products = branded.get("products", [])
print(f"Brand-filtered results: {branded.get('total', 0)}")
for p in branded_products:
    print(f"  ${p.get('price', 0):>6.2f} | score: {p.get('matchScore', 0):.3f} | {p.get('category', '-')}")

show_query_and_results(query_image_url, branded_products, query_title="Branded Matches")

In [ ]:
# Image search with retailer filter
retailer_results = image_search(
    query_image_url,
    limit=5,
    sites=["nordstrom.com", "farfetch.com"],
)

retailer_products = retailer_results.get("products", [])
print(f"From Nordstrom + Farfetch: {retailer_results.get('total', 0)} results")
for p in retailer_products:
    print(f"  ${p.get('price', 0):>6.2f} | score: {p.get('matchScore', 0):.3f} | {p.get('category', '-')}")

---
## Task 6: "Shop the Look" Pipeline

A common production pattern: given an outfit photo, find shoppable products for each visible item. We combine bounding box cropping with text guidance to search different regions of the image.

In [ ]:
# Get an outfit image
outfit_items = text_search("women street style outfit", top_k=3)
look_url = outfit_items[0].get("imageUrl", query_image_url)

try:
    look_img = load_image(look_url)
    w, h = look_img.size
except Exception:
    w, h = 400, 600

# Define regions for different clothing items
regions = [
    {"name": "Top",    "bbox": [0, 0, w, h // 3],          "desc": "top blouse shirt"},
    {"name": "Bottom", "bbox": [0, h // 3, w, 2 * h // 3], "desc": "skirt pants trousers"},
    {"name": "Shoes",  "bbox": [0, 2 * h // 3, w, h],      "desc": "shoes boots heels"},
]

print(f"Image: {w}x{h}")
print(f"Searching {len(regions)} regions...\n")

shop_the_look = {}
for region in regions:
    result = image_search(
        look_url,
        limit=3,
        bbox=region["bbox"],
        imageDescription=region["desc"],
    )
    products = result.get("products", [])
    shop_the_look[region["name"]] = products
    print(f"{region['name']:>8}: {len(products)} matches")
    for p in products:
        print(f"           {p.get('category', '-'):>12} | "
              f"${p.get('price', 0):.2f} | score: {p.get('matchScore', 0):.3f}")

In [ ]:
# Visualize the "Shop the Look" results
n_regions = len(regions)
fig, axes = plt.subplots(n_regions, 4, figsize=(16, 4 * n_regions))

for row, region in enumerate(regions):
    # Query region with bbox overlay
    try:
        rimg = load_image(look_url)
        draw = ImageDraw.Draw(rimg)
        draw.rectangle(region["bbox"], outline="red", width=3)
        axes[row, 0].imshow(rimg)
    except Exception:
        axes[row, 0].text(0.5, 0.5, "N/A", ha="center", va="center")
    axes[row, 0].set_title(f"{region['name']}\nbbox: {region['bbox']}", fontsize=9, fontweight="bold")
    axes[row, 0].axis("off")

    # Matched products
    products = shop_the_look.get(region["name"], [])
    for j in range(3):
        ax = axes[row, j + 1]
        if j < len(products):
            p = products[j]
            try:
                ax.imshow(load_image(p.get("imageUrl", "")))
            except Exception:
                pass
            price = p.get("price", 0)
            score = p.get("matchScore", 0)
            ax.set_title(f"${price:.2f} | score: {score:.3f}", fontsize=8)
        else:
            ax.text(0.5, 0.5, "No match", ha="center", va="center")
        ax.axis("off")

plt.suptitle("Shop the Look", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrated how to use APIClaw's Fashion Image Search API for visual product discovery.

### APIs Used

| Endpoint | Purpose | Cost |
|----------|---------|------|
| `POST /openapi/v2/model/fashion-image-search` | Visual similarity product search | 1 credit/request |
| `POST /openapi/v2/model/fashion-product-search` | Text search (for demo images) | 1 credit/request |

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| `imageUrl` | HTTPS URL of the query image |
| `limit` | Max results (1-50) |
| `offset` | Pagination offset |
| `bbox` | Crop region `[x1, y1, x2, y2]` in pixels |
| `imageDescription` | Text to compose with image for guided search |
| `brands` | Brand name filter |
| `priceMin` / `priceMax` | Price range in USD |
| `sites` | Retailer domain filter |

### Production Patterns

- **"Find Similar"**: Basic image search → visually similar products
- **"Shop the Look"**: Bounding box + text guidance → per-item product matches
- **"Affordable Dupes"**: Image search + price filter → budget alternatives
- **"Brand Match"**: Image search + brand filter → find specific brand alternatives

### Next Steps

- Try the [Fashion Product Search](https://colab.research.google.com/github/SerendipityOneInc/APIClaw-Notebook/blob/main/e-commerce/fashion_product_search_demo.ipynb) notebook for text-based discovery
- Combine with FashionSigLIP2 embeddings for custom reranking
- Get your API key at [apiclaw.io](https://apiclaw.io)